**Install Required Libraries**

In [1]:
!pip install transformers datasets torch seqeval tqdm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
!pip install datasets==3.6.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 9.4 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


# **Import Libraries**

In [3]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW  # AdamW is now in torch.optim
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from datasets import load_dataset
import numpy as np
from tqdm import tqdm
from seqeval.metrics import f1_score as ner_f1_score

In [4]:
# Set random seed for reproducibility

torch.manual_seed(42)
np.random.seed(42)


In [5]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


# **Load Tokenizer**


**Dataset Preparation - Tokenization**  
**Description:** Loading the BERT tokenizer that will be used consistently for both NER and QA tasks.

In [6]:
model_name = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
print(f"Tokenizer loaded: {model_name}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizer loaded: bert-base-uncased


# **NER Dataset Class**

**Dataset Preparation - NER (CoNLL-2003)**  
**Description:** Creating NER dataset class with proper token-level tag alignment using IOB format. Handles subword tokenization by using -100 for subsequent sub-tokens to ignore in loss calculation.

In [7]:
class NERDataset(Dataset):
    """CoNLL-2003 NER Dataset with IOB format label alignment"""

    def __init__(self, dataset, tokenizer, max_length=128):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.max_length = max_length

        # CoNLL-2003 has 9 labels (O + 8 entity types in IOB format)
        self.label_list = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG',
                          'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        tokens = item['tokens']
        ner_tags = item['ner_tags']

        # Tokenize with subword handling
        tokenized = self.tokenizer(
            tokens,
            is_split_into_words=True,
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )

        # Align labels: -100 for special tokens and subsequent subword tokens
        word_ids = tokenized.word_ids(batch_index=0)
        aligned_labels = []
        previous_word_idx = None

        for word_idx in word_ids:
            if word_idx is None:
                # Special tokens: [CLS], [SEP], [PAD]
                aligned_labels.append(-100)
            elif word_idx != previous_word_idx:
                # First token of a word - assign actual label
                aligned_labels.append(ner_tags[word_idx])
            else:
                # Subsequent subword tokens - use -100 to ignore in loss
                aligned_labels.append(-100)
            previous_word_idx = word_idx

        return {
            'input_ids': tokenized['input_ids'].squeeze(0),
            'attention_mask': tokenized['attention_mask'].squeeze(0),
            'labels': torch.tensor(aligned_labels, dtype=torch.long),
            'task_name': 'ner'
        }

print("NER Dataset class created")

NER Dataset class created


# **QA Dataset Class**

**Dataset Preparation - QA (SQuAD)**  
**Description:** Creating QA dataset class with answer span extraction. Input format: [CLS] question [SEP] context [SEP]. Adjusts answer span indices to tokenized positions.

In [8]:
class QADataset(Dataset):
    """SQuAD QA Dataset with answer span alignment"""

    def __init__(self, dataset, tokenizer, max_length=384):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        # Handle different possible column names in SQuAD
        question = item.get('question', item.get('questions', ''))
        context = item.get('context', item.get('contexts', ''))
        answers = item.get('answers', {'text': [], 'answer_start': []})


        # Tokenize: [CLS] question [SEP] context [SEP]
        tokenized = self.tokenizer(
            question,
            context,
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )

        # Find answer span in tokenized input
        if len(answers['text']) > 0:
            answer_text = answers['text'][0]
            answer_start_char = answers['answer_start'][0]
            answer_end_char = answer_start_char + len(answer_text)

            # Map character positions to token positions
            start_token = None
            end_token = None

            for i in range(len(tokenized['input_ids'][0])):
                span = tokenized.token_to_chars(0, i)
                if span is None:
                    continue

                if span.start <= answer_start_char < span.end and start_token is None:
                    start_token = i
                if span.start < answer_end_char <= span.end:
                    end_token = i
                    break

            # If answer not found in truncated context, set to [CLS] position
            if start_token is None or end_token is None:
                start_token = 0
                end_token = 0
        else:
            # No answer case (SQuAD v2 style)
            start_token = 0
            end_token = 0

        return {
            'input_ids': tokenized['input_ids'].squeeze(0),
            'attention_mask': tokenized['attention_mask'].squeeze(0),
            'start_positions': torch.tensor(start_token, dtype=torch.long),
            'end_positions': torch.tensor(end_token, dtype=torch.long),
            'task_name': 'qa'
        }

print("QA Dataset class created")

QA Dataset class created


# **Multi-Task DataLoader**

**Dataset Preparation - Unified DataLoader**  
**Description:** Custom DataLoader implementing Round-Robin sampling strategy between NER and QA batches as specified in instructions.

In [9]:
class MultiTaskDataLoader:
    """
    Custom DataLoader for Multi-Task Learning
    Implements Round-Robin sampling strategy
    """

    def __init__(self, ner_dataset, qa_dataset, batch_size=16, shuffle=True):
        self.ner_loader = DataLoader(ner_dataset, batch_size=batch_size, shuffle=shuffle)
        self.qa_loader = DataLoader(qa_dataset, batch_size=batch_size, shuffle=shuffle)
        self.batch_size = batch_size

    def __iter__(self):
        ner_iter = iter(self.ner_loader)
        qa_iter = iter(self.qa_loader)

        # Round-robin between NER and QA tasks
        while True:
            try:
                # Yield NER batch
                yield next(ner_iter)
            except StopIteration:
                break

            try:
                # Yield QA batch
                yield next(qa_iter)
            except StopIteration:
                break

    def __len__(self):
        return len(self.ner_loader) + len(self.qa_loader)

print("MultiTaskDataLoader class created")

MultiTaskDataLoader class created


# **Load Datasets**

**Dataset Selection**  
**Description:** Loading CoNLL-2003 (NER) and SQuAD (QA) datasets from Hugging Face as specified in instructions


In [10]:
from datasets import load_dataset

print("Loading CoNLL-2003 dataset...")
ner_dataset = load_dataset("conll2003", trust_remote_code=True)

train_ner = NERDataset(ner_dataset["train"], tokenizer)
val_ner = NERDataset(ner_dataset["validation"], tokenizer)
test_ner = NERDataset(ner_dataset["test"], tokenizer)

print(f"  ✓ NER Train: {len(train_ner)} examples")
print(f"  ✓ NER Validation: {len(val_ner)} examples")
print(f"  ✓ NER Test: {len(test_ner)} examples")

# Load SQuAD dataset with SUBSET to avoid RAM crash
print("\nLoading SQuAD dataset (using subset for RAM efficiency)...")
qa_dataset = load_dataset('squad')

# Use only 5% of the data to fit in RAM
train_size = int(len(qa_dataset["train"]) * 0.05)  # ~4,400 examples
val_size = int(len(qa_dataset["validation"]) * 0.05)  # ~530 examples

train_qa = QADataset(qa_dataset["train"].select(range(train_size)), tokenizer)
val_qa = QADataset(qa_dataset["validation"].select(range(val_size)), tokenizer)

print(f"  ✓ QA Train: {len(train_qa)} examples (subset)")
print(f"  ✓ QA Validation: {len(val_qa)} examples (subset)")

Loading CoNLL-2003 dataset...


README.md: 0.00B [00:00, ?B/s]

conll2003.py: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

...

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

  ✓ QA Train: 4379 examples (subset)
  ✓ QA Validation: 528 examples (subset)


# **Create DataLoaders**


**Description:** Creating multi-task training loader and separate validation loaders for each task.

In [11]:
batch_size = 8

# Multi-task training loader (Round-Robin)
train_loader = MultiTaskDataLoader(train_ner, train_qa, batch_size=batch_size, shuffle=True)

# Separate validation loaders for evaluation
val_ner_loader = DataLoader(val_ner, batch_size=batch_size, shuffle=False)
val_qa_loader = DataLoader(val_qa, batch_size=batch_size, shuffle=False)

print(f"DataLoaders created with batch_size={batch_size}")
print(f"Total training batches: {len(train_loader)}")

DataLoaders created with batch_size=8
Total training batches: 2304


# **Multi-Task Model Architecture**

  
**Description:** Defining the MultiTaskModel with:
- Shared Encoder: BERT (shared across both tasks)
- NER Head: Linear layer mapping hidden_size → num_labels
- QA Head: Two linear layers for start/end logits

In [12]:


class MultiTaskModel(nn.Module):


    def __init__(self, model_name='bert-base-uncased', num_ner_labels=9):
        super(MultiTaskModel, self).__init__()

        # Shared Transformer Encoder (from 2.2)
        self.encoder = AutoModel.from_pretrained(model_name)
        self.hidden_size = self.encoder.config.hidden_size

        # Task-Specific Heads (from 2.2)
        # NER Head: Maps hidden states to number of tags
        self.ner_head = nn.Linear(self.hidden_size, num_ner_labels)

        # QA Head: Two layers for start and end logits
        self.qa_start_head = nn.Linear(self.hidden_size, 1)
        self.qa_end_head = nn.Linear(self.hidden_size, 1)

        # Dropout for regularization
        self.dropout = nn.Dropout(0.1)

    def forward(self, input_ids, attention_mask, task_name, labels=None,
                start_positions=None, end_positions=None):
        """
        Forward pass with task-specific branching (from 2.2)
        """

        # 1. Shared Encoder Pass
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state
        sequence_output = self.dropout(sequence_output)

        # 2. Task-Specific Head Processing
        if task_name == 'ner':
            # NER Head
            ner_logits = self.ner_head(sequence_output)

            loss = None
            if labels is not None:
                # Cross-Entropy Loss (from Part 1.2)
                loss_fct = nn.CrossEntropyLoss()
                loss = loss_fct(ner_logits.view(-1, ner_logits.size(-1)), labels.view(-1))

            return {'loss': loss, 'logits': ner_logits}

        elif task_name == 'qa':
            # QA Head
            start_logits = self.qa_start_head(sequence_output).squeeze(-1)
            end_logits = self.qa_end_head(sequence_output).squeeze(-1)

            loss = None
            if start_positions is not None and end_positions is not None:
                # Cross-Entropy Loss for start + end (from Part 1.2)
                loss_fct = nn.CrossEntropyLoss()
                start_loss = loss_fct(start_logits, start_positions)
                end_loss = loss_fct(end_logits, end_positions)
                loss = (start_loss + end_loss) / 2

            return {'loss': loss, 'start_logits': start_logits, 'end_logits': end_logits}

        else:
            raise ValueError(f"Unknown task_name: {task_name}")

print("MultiTaskModel class created")

MultiTaskModel class created


# **Initialize Model**

**Description:** Creating the model instance with BERT as shared encoder and task-specific heads.


In [13]:
# Initialize Multi-Task Model
model = MultiTaskModel(model_name='bert-base-uncased', num_ner_labels=9)
model.to(device)

print("Model initialized and moved to device")
print(f"  - Shared Encoder: bert-base-uncased")
print(f"  - Hidden Size: {model.hidden_size}")
print(f"  - NER Head: Linear({model.hidden_size} → 9)")
print(f"  - QA Head: Two Linear layers ({model.hidden_size} → 1)")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Model initialized and moved to device
  - Shared Encoder: bert-base-uncased
  - Hidden Size: 768
  - NER Head: Linear(768 → 9)
  - QA Head: Two Linear layers (768 → 1)


# **Setup Optimizer and Scheduler**

**Custom Training Loop - Initialization**  
**Description:** Setting up AdamW optimizer and linear learning rate scheduler with warmup


In [14]:
# Training hyperparameters
num_epochs = 3
learning_rate = 2e-5

# AdamW optimizer (from 2.3)
optimizer = AdamW(model.parameters(), lr=learning_rate)

# Learning rate scheduler with warmup
total_steps = len(train_loader) * num_epochs
warmup_steps = int(0.1 * total_steps)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print(f"Optimizer and scheduler configured")
print(f"  - Total training steps: {total_steps}")
print(f"  - Warmup steps: {warmup_steps}")

Optimizer and scheduler configured
  - Total training steps: 6912
  - Warmup steps: 691


# **Training Loop with Multi-Task Loss**

**Custom Training Loop - Main Loop**  
**Description:** Custom training loop implementing weighted multi-task loss:  
**L_MTL = λ_NER × L_NER + λ_QA × L_QA** (from Part 1.2)


In [15]:
# Loss weights
lambda_ner = 1.0
lambda_qa = 1.0

# Training loop
for epoch in range(num_epochs):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch + 1}/{num_epochs}")
    print(f"{'='*60}")

    model.train()
    train_loss = 0
    ner_loss_sum = 0
    qa_loss_sum = 0
    ner_count = 0
    qa_count = 0

    progress_bar = tqdm(train_loader, desc="Training")

    for batch in progress_bar:
        # Move batch to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        task_name = batch['task_name'][0]

        # Task-specific forward pass
        if task_name == 'ner':
            labels = batch['labels'].to(device)
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                task_name='ner',
                labels=labels
            )
            # Apply λ_NER weight (from Part 1.2)
            loss = outputs['loss'] * lambda_ner
            ner_loss_sum += outputs['loss'].item()
            ner_count += 1

        else:  # qa
            start_positions = batch['start_positions'].to(device)
            end_positions = batch['end_positions'].to(device)
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                task_name='qa',
                start_positions=start_positions,
                end_positions=end_positions
            )
            # Apply λ_QA weight (from Part 1.2)
            loss = outputs['loss'] * lambda_qa
            qa_loss_sum += outputs['loss'].item()
            qa_count += 1

        # Backward pass and optimization (from 2.3)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        train_loss += loss.item()
        progress_bar.set_postfix({'loss': loss.item()})

    # Print epoch statistics
    avg_train_loss = train_loss / len(train_loader)
    avg_ner_loss = ner_loss_sum / ner_count if ner_count > 0 else 0
    avg_qa_loss = qa_loss_sum / qa_count if qa_count > 0 else 0

    print(f"\nEpoch {epoch + 1} Results:")
    print(f"  - Average Training Loss: {avg_train_loss:.4f}")
    print(f"  - NER Loss: {avg_ner_loss:.4f}")
    print(f"  - QA Loss: {avg_qa_loss:.4f}")

print("\n" + "="*60)
print("Training completed!")
print("="*60)


Epoch 1/3


Training:  48%|████▊     | 1097/2304 [07:24<08:09,  2.47it/s, loss=0.137]



Epoch 1 Results:
  - Average Training Loss: 1.0182
  - NER Loss: 0.4632
  - QA Loss: 3.8168

Epoch 2/3


Training:  48%|████▊     | 1097/2304 [07:25<08:10,  2.46it/s, loss=0.0451]



Epoch 2 Results:
  - Average Training Loss: 0.4734
  - NER Loss: 0.0762
  - QA Loss: 1.9140

Epoch 3/3


Training:  48%|████▊     | 1097/2304 [07:26<08:10,  2.46it/s, loss=0.0823]


Epoch 3 Results:
  - Average Training Loss: 0.3068
  - NER Loss: 0.0541
  - QA Loss: 1.2357

Training completed!


# **NER Evaluation Function**

Evaluating NER task using Micro-averaged F1-Score

In [16]:
def evaluate_ner(model, dataloader, device):
    """Evaluate NER using F1-Score (Micro-averaged)"""
    model.eval()

    ner_labels = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG',
                  'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

    true_labels = []
    pred_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating NER"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                task_name='ner'
            )

            predictions = torch.argmax(outputs['logits'], dim=-1)

            # Convert to label strings (ignoring -100)
            for i in range(labels.size(0)):
                true_seq = []
                pred_seq = []
                for j in range(labels.size(1)):
                    if labels[i, j] != -100:
                        true_seq.append(ner_labels[labels[i, j]])
                        pred_seq.append(ner_labels[predictions[i, j]])
                true_labels.append(true_seq)
                pred_labels.append(pred_seq)

    # F1-Score (Micro-averaged) from Part 3.1
    f1 = ner_f1_score(true_labels, pred_labels)
    return f1

print("NER evaluation function created")

NER evaluation function created


# **QA Evaluation Function**

**Evaluation - QA F1 and Exact Match**  
**Description:** Evaluating QA task using F1-Score (token overlap) and Exact Match



In [17]:
def evaluate_qa(model, dataloader, device):
    """Evaluate QA using F1-Score and Exact Match (EM)"""
    model.eval()

    total_f1 = 0
    total_em = 0
    count = 0

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating QA"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            start_positions = batch['start_positions'].to(device)
            end_positions = batch['end_positions'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                task_name='qa'
            )

            # Get predictions
            pred_start = torch.argmax(outputs['start_logits'], dim=-1)
            pred_end = torch.argmax(outputs['end_logits'], dim=-1)

            # Calculate metrics for each example
            for i in range(input_ids.size(0)):
                # Predicted answer tokens
                start = pred_start[i].item()
                end = pred_end[i].item()
                if end < start:
                    end = start
                pred_tokens = set(range(start, end + 1))

                # True answer tokens
                true_start = start_positions[i].item()
                true_end = end_positions[i].item()
                true_tokens = set(range(true_start, true_end + 1))

                # Calculate F1 (token overlap) from Part 3.1
                if len(pred_tokens) == 0 or len(true_tokens) == 0:
                    f1 = float(pred_tokens == true_tokens)
                else:
                    common = len(pred_tokens & true_tokens)
                    if common == 0:
                        f1 = 0
                    else:
                        precision = common / len(pred_tokens)
                        recall = common / len(true_tokens)
                        f1 = 2 * precision * recall / (precision + recall)

                # Calculate Exact Match from Part 3.1
                em = float(pred_tokens == true_tokens)

                total_f1 += f1
                total_em += em
                count += 1

    return total_f1 / count, total_em / count

print("QA evaluation function created")

QA evaluation function created


# **Run Final Evaluation**

**Description:** Running evaluation on both tasks separately on their dedicated test sets

In [18]:
print("="*60)
print("FINAL EVALUATION")
print("="*60)

# Evaluate NER
print("\n--- NER Evaluation (CoNLL-2003) ---")
ner_f1 = evaluate_ner(model, val_ner_loader, device)
print(f"NER F1-Score (Micro-averaged): {ner_f1:.4f}")

# Evaluate QA
print("\n--- QA Evaluation (SQuAD) ---")
qa_f1, qa_em = evaluate_qa(model, val_qa_loader, device)
print(f"QA F1-Score (Token Overlap): {qa_f1:.4f}")
print(f"QA Exact Match (EM): {qa_em:.4f}")

print("\n" + "="*60)
print("Evaluation completed!")
print("="*60)

FINAL EVALUATION

--- NER Evaluation (CoNLL-2003) ---


Evaluating NER: 100%|██████████| 407/407 [00:34<00:00, 11.92it/s]


NER F1-Score (Micro-averaged): 0.9352

--- QA Evaluation (SQuAD) ---


Evaluating QA: 100%|██████████| 66/66 [00:13<00:00,  5.07it/s]

QA F1-Score (Token Overlap): 0.5148
QA Exact Match (EM): 0.4053

Evaluation completed!


In [19]:
# Save the trained model
torch.save(model.state_dict(), 'multitask_ner_qa_model.pt')
print("Model saved as 'multitask_ner_qa_model.pt'")

Model saved as 'multitask_ner_qa_model.pt'
